# Домашнее задание 3
## Краулинг коллекции Метрополитен-музея

**Что делаем.** Собираем коллекцию европейской живописи Метрополитен-музея:
находим 125 «избранных» работ, обходим страницу каждой картины, достаём
название, автора, дату, технику, размеры и другие сведения, складываем всё
в таблицу pandas и отвечаем на вопросы о данных. Дополнительные задания — про
Selenium: смотрим, как сайт музея Виктории и Альберта рисует результаты
поиска через JavaScript, и собираем данные через браузер.

**Оценивание.** Максимум за основную часть — **8,5 баллов**. Дополнительное задание (Selenium) даёт **+1,5 балла**.

**Что сдаём.** Ссылку на файл (не папку) с заполненным ноутбуком `фамилия_hw3.ipynb`. Все ячейки с кодом
запущены.


In [1]:
%pip install requests beautifulsoup4 pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: C:\Users\honor\AppData\Local\Python\pythoncore-3.13-64\python.exe -m pip install --upgrade pip


## Часть 1. Сбор данных: requests + BeautifulSoup — 6 баллов

Коллекция Метрополитен-музея открыта на
[metmuseum.org/art/collection](https://www.metmuseum.org/art/collection).

Мы возьмём не всю коллекцию (там сотни тысяч предметов), а небольшую
понятную выборку:

- `department=11` — отдел **European Paintings** (европейская живопись);
- `showOnly=highlights` — только работы из раздела «избранное».

> **Про пагинацию.** Кнопка «Next Page» на сайте работает через JavaScript,
> но у страницы есть параметр `offset`. Первая страница — `offset=0`,
> вторая — `offset=42`, третья — `offset=84`. 

In [3]:
import re
import time
import requests
import pandas as pd

from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE_URL = "https://www.metmuseum.org"
SEARCH_URL = "https://www.metmuseum.org/art/collection/search"

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0"
})

def get_soup(url, params=None, pause=1):
    response = session.get(url, params=params, timeout=30)
    response.raise_for_status()
    time.sleep(pause)
    return BeautifulSoup(response.text, "html.parser"), response

In [4]:
soup, response = get_soup(
    SEARCH_URL,
    params={
        "department": "European Paintings",
        "showOnly": "highlights",
        "offset": 0
    }
)

print(response.status_code)
print(response.url)

200
https://www.metmuseum.org/art/collection/search?department=European+Paintings&showOnly=highlights&offset=0


### Задание 1. Разобрать одну работу — 1 балл

Откройте любую карточку и посмотрите на блок **Artwork Details** (в HTML у него `id="artwork-details"`).

Напишите функцию `parse_artwork()`, которая возвращает словарь
из десяти ключей:

`title`, `artist`, `date`, `medium`, `dimensions`, `classification`,
`credit_line`, `object_number`, `department`, `url`.

Проверьте функцию на одной работе.

In [6]:
def parse_artwork(url):
    soup, response = get_soup(url, pause=0.5)

    details = soup.find(id="artwork-details")

    if details is None:
        raise ValueError("Artwork Details не найден")

    fields = {
        "Title": "title",
        "Artist": "artist",
        "Date": "date",
        "Medium": "medium",
        "Dimensions": "dimensions",
        "Classification": "classification",
        "Credit Line": "credit_line",
        "Object Number": "object_number",
        "Department": "department"
    }

    data = {}

    for dt in details.find_all("dt"):
        dd = dt.find_next_sibling("dd")

        if dd:
            label = dt.get_text(" ", strip=True).rstrip(":")
            value = dd.get_text(" ", strip=True)

            if label in fields:
                data[fields[label]] = value

    if not data.get("title"):
        heading = soup.find("h1")
        if heading:
            data["title"] = heading.get_text(" ", strip=True)

    result = {}

    for key in [
        "title", "artist", "date", "medium", "dimensions",
        "classification", "credit_line", "object_number", "department"
    ]:
        result[key] = data.get(key, "")

    result["url"] = response.url

    return result

In [9]:
def parse_artwork(url):
    soup, response = get_soup(url, pause=0.5)

    details = soup.find(id="artwork-details")

    if details is None:
        raise ValueError("Artwork Details нт")

    data = {}

    fields = {
        "Title": "title",
        "Artist": "artist",
        "Date": "date",
        "Medium": "medium",
        "Dimensions": "dimensions",
        "Classification": "classification",
        "Credit Line": "credit_line",
        "Object Number": "object_number",
        "Curatorial Department": "department"
    }

    for item in details.select("ul li"):
        label = item.find("strong")
        value = item.find(["span", "a"])

        if label and value:
            key = fields.get(label.get_text(strip=True).rstrip(":"))

            if key:
                data[key] = value.get_text(" ", strip=True)

    result = {
        "title": data.get("title", ""),
        "artist": data.get("artist", ""),
        "date": data.get("date", ""),
        "medium": data.get("medium", ""),
        "dimensions": data.get("dimensions", ""),
        "classification": data.get("classification", ""),
        "credit_line": data.get("credit_line", ""),
        "object_number": data.get("object_number", ""),
        "department": data.get("department", ""),
        "url": response.url
    }

    return result

In [10]:
links = soup.select('a[href*="/art/collection/search/"]')

test_url = urljoin(BASE_URL, links[0]["href"])

test_artwork = parse_artwork(test_url)

test_artwork

{'title': 'Saint John on Patmos',
 'artist': 'Hans Baldung (called Hans Baldung Grien) (German, Schwäbisch Gmünd (?) 1484/85–1545 Strasbourg)',
 'date': 'ca. 1511',
 'medium': 'Oil, gold, and white metal on spruce',
 'dimensions': 'Overall 35 1/4 x 30 1/4 in. (89.5 x 76.8 cm); painted surface 34 3/8 x 29 3/4 in. (87.3 x 75.6 cm)',
 'classification': 'Paintings',
 'credit_line': 'Purchase, Rogers and Fletcher Funds; The Vincent Astor Foundation, The Dillon Fund, The Charles Engelhard Foundation, Lawrence A. Fleischman, Mrs. Henry J. Heinz II, The Willard T. C. Johnson Foundation Inc., Reliance Group Holdings Inc., Baron H. H. Thyssen-Bornemisza, and Mr. and Mrs. Charles Wrightsman Gifts; Joseph Pulitzer Bequest; special funds; and other gifts and bequests, by exchange, 1983',
 'object_number': '1983.451',
 'department': 'European Paintings',
 'url': 'https://www.metmuseum.org/art/collection/search/435600'}

### Задание 2. Собрать адреса всех работ — 2 балла

Скачайте страницы поиска (`offset=0`, `42` и тд) и соберите адреса
всех карточек.

Не забудьте превратить относительный адрес в абсолютный через `urljoin(BASE_URL, href)`.

In [11]:
offsets = [0, 42, 84]
artwork_urls = []

for offset in offsets:
    soup, response = get_soup(
        SEARCH_URL,
        params={
            "department": "European Paintings",
            "showOnly": "highlights",
            "offset": offset
        }
    )

    links = soup.select('a[href*="/art/collection/search/"]')

    for link in links:
        href = link.get("href")

        if href:
            url = urljoin(BASE_URL, href)

            if re.search(r"/art/collection/search/\d+", url):
                artwork_urls.append(url)

artwork_urls = list(dict.fromkeys(artwork_urls))

print("Кол-во адресов:", len(artwork_urls))
print(artwork_urls[:5])

Кол-во адресов: 125
['https://www.metmuseum.org/art/collection/search/435600', 'https://www.metmuseum.org/art/collection/search/435621', 'https://www.metmuseum.org/art/collection/search/435641', 'https://www.metmuseum.org/art/collection/search/435702', 'https://www.metmuseum.org/art/collection/search/435728']


### Задание 3. Обойти все работы и собрать записи — 2 балла

Теперь у вас есть список из 125 адресов. Напишите краулер, который:

1. проходит по адресам и скачивает каждую страницу;
2. разбирает её через `parse_artwork`;
3. добавляет полученный словарь в общий список `records`

Проверьте, что `len(records) == 125`, и выведите первую запись.

In [13]:
records = []

for i, url in enumerate(artwork_urls, start=1):
    try:
        artwork = parse_artwork(url)
        records.append(artwork)
        print(i, artwork["title"])

    except Exception as error:
        print("Ошибка:", url, error)

    time.sleep(1)

1 Saint John on Patmos
2 Joan of Arc
3 Madonna and Child
4 The Horse Fair
5 The Last Communion of Saint Jerome
6 The Toilette of Venus
7 Portrait of a Young Man
8 The Harvesters
9 The Crucifixion with the Virgin and Saint John
10 The Love Song
11 Piazza San Marco
12 The Musicians
13 The Meditation on the Passion
14 The Coronation of the Virgin
15 The Card Players
16 Madame Cézanne (Hortense Fiquet, 1850–1922) in a Red Dress
17 Still Life with Apples and a Pot of Primroses
18 Soap Bubbles
19 Portrait of a Carthusian
20 The Trojan Women Setting Fire to Their Fleet
21 Hagar in the Wilderness
22 Sibylle
23 Woman with a Parrot
24 The Third-Class Carriage
25 The Rest on the Flight into Egypt
26 The Death of Socrates
27 Antoine Laurent Lavoisier (1743–1794) and Marie Anne Lavoisier (Marie Anne Pierrette Paulze, 1758–1836)
28 A Woman Seated beside a Vase of Flowers (Madame Paul Valpinçon?)
29 Woman Having Her Hair Combed
30 Virgin and Child with Saint Anne
31 Lucas van Uffel (died 1637)
32 The

### Задание 4. Почистить данные и собрать DataFrame — 0,5 балла
Почистите данные:

1. Оставьте имя художника без национальности и дат.

2. Избавьтесь от названий полей, например, `'Title: Wheat Field with Cypresses'` нужно убрать `'Title:'`.

Соберите `df = pd.DataFrame(records)`, добавьте новые колонки и проверьте
   таблицу через `df.info()` и `df.head()`.

In [16]:
def clean_artist(value):
    if not isinstance(value, str):
        return ""

    value = re.sub(r"\([^)]*\)", "", value)
    value = re.sub(r",?\s*\d{3,4}\s*[-–—]\s*\d{0,4}", "", value)

    return value.strip(" ,;")


def clean_text(value):
    if not isinstance(value, str):
        return ""

    value = re.sub(
        r"^(Title|Artist|Date|Medium|Dimensions|Classification|Credit Line|Object Number|Department):\s*",
        "",
        value,
        flags=re.IGNORECASE
    )

    return value.strip()


df = pd.DataFrame(records)

df["title_clean"] = df["title"].apply(clean_text)
df["artist_short"] = df["artist"].apply(clean_artist)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 125 entries, 0 to 124
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   title           125 non-null    str  
 1   artist          125 non-null    str  
 2   date            125 non-null    str  
 3   medium          125 non-null    str  
 4   dimensions      125 non-null    str  
 5   classification  125 non-null    str  
 6   credit_line     125 non-null    str  
 7   object_number   125 non-null    str  
 8   department      125 non-null    str  
 9   url             125 non-null    str  
 10  title_clean     125 non-null    str  
 11  artist_short    125 non-null    str  
dtypes: str(12)
memory usage: 11.8 KB


In [17]:
display(df.head())

,title,artist,date,medium,dimensions,classification,credit_line,object_number,department,url,title_clean,artist_short
0,Saint John on Patmos,Hans Baldung (called Hans Baldung Grien) (Germ...,ca. 1511,"Oil, gold, and white metal on spruce",Overall 35 1/4 x 30 1/4 in. (89.5 x 76.8 cm); ...,Paintings,"Purchase, Rogers and Fletcher Funds; The Vince...",1983.451,European Paintings,https://www.metmuseum.org/art/collection/searc...,Saint John on Patmos,Hans Baldung 1484/85–1545 Strasbourg)
1,Joan of Arc,"Jules Bastien-Lepage (French, Damvillers 1848–...",1879,Oil on canvas,100 x 110 in. (254 x 279.4 cm),Paintings,"Gift of Erwin Davis, 1889",89.21.1,European Paintings,https://www.metmuseum.org/art/collection/searc...,Joan of Arc,Jules Bastien-Lepage
2,Madonna and Child,"Giovanni Bellini (Italian, Venice, 1424/26–151...",late 1480s,Oil on wood,35 x 28 in. (88.9 x 71.1 cm),Paintings,"Rogers Fund, 1908",08.183.1,European Paintings,https://www.metmuseum.org/art/collection/searc...,Madonna and Child,Giovanni Bellini
3,The Horse Fair,"Rosa Bonheur (French, Bordeaux 1822–1899 Thomery)",1852–55,Oil on canvas,96 1/4 x 199 1/2 in. (244.5 x 506.7 cm),Paintings,"Gift of Cornelius Vanderbilt, 1887",87.25,European Paintings,https://www.metmuseum.org/art/collection/searc...,The Horse Fair,Rosa Bonheur
4,The Last Communion of Saint Jerome,Botticelli (Alessandro di Mariano Filipepi) (I...,early 1490s,Tempera and gold on wood,13 1/2 x 10 in. (34.3 x 25.4 cm),Paintings,"Bequest of Benjamin Altman, 1913",14.40.642,European Paintings,https://www.metmuseum.org/art/collection/searc...,The Last Communion of Saint Jerome,Botticelli


### Задание 5*. Получше почистить данные — максимум 1 балл

Попробуйте почистить другие поля: даты, размер. Может быть, что-то ещё требует другого формата хранения? Если да, обоснуйте и попробуйте реализовать. Отталкивайтесь от того, на какие вопросы будут отвечать данные.

In [18]:
def extract_year(value):
    if not isinstance(value, str):
        return pd.NA

    match = re.search(r"\b(?:1[0-9]{3}|20[0-2][0-9])\b", value)

    if match:
        return int(match.group())

    return pd.NA


df["year"] = df["date"].apply(extract_year).astype("Int64")

In [19]:
def extract_dimensions(value):
    if not isinstance(value, str):
        return pd.Series([pd.NA, pd.NA])

    numbers = re.findall(r"(\d+(?:\.\d+)?)\s*cm", value, re.I)

    if len(numbers) >= 2:
        return pd.Series([float(numbers[0]), float(numbers[1])])

    return pd.Series([pd.NA, pd.NA])


df[["height_cm", "width_cm"]] = df["dimensions"].apply(
    extract_dimensions
)

df["area_cm2"] = df["height_cm"] * df["width_cm"]

display(df[[
    "title_clean", "date", "year",
    "dimensions", "height_cm", "width_cm", "area_cm2"
]].head(10))

,title_clean,date,year,dimensions,height_cm,width_cm,area_cm2
0,Saint John on Patmos,ca. 1511,1511,Overall 35 1/4 x 30 1/4 in. (89.5 x 76.8 cm); ...,76.8,75.6,5806.08
1,Joan of Arc,1879,1879,100 x 110 in. (254 x 279.4 cm),<NA>,<NA>,<NA>
2,Madonna and Child,late 1480s,<NA>,35 x 28 in. (88.9 x 71.1 cm),<NA>,<NA>,<NA>
3,The Horse Fair,1852–55,1852,96 1/4 x 199 1/2 in. (244.5 x 506.7 cm),<NA>,<NA>,<NA>
4,The Last Communion of Saint Jerome,early 1490s,<NA>,13 1/2 x 10 in. (34.3 x 25.4 cm),<NA>,<NA>,<NA>
5,The Toilette of Venus,1751,1751,42 5/8 x 33 1/2 in. (108.3 x 85.1 cm),<NA>,<NA>,<NA>
6,Portrait of a Young Man,1530s,<NA>,37 5/8 x 29 1/2 in. (95.6 x 74.9 cm),<NA>,<NA>,<NA>
7,The Harvesters,1565,1565,"Overall, including added strips at top, bottom...",162.0,159.5,25839.0
8,The Crucifixion with the Virgin and Saint John,ca. 1624–25,1624,61 in. × 40 1/4 in. (154.9 × 102.2 cm),<NA>,<NA>,<NA>
9,The Love Song,1868–77,1868,45 x 61 3/8 in. (114.3 x 155.9 cm),<NA>,<NA>,<NA>


In [21]:
print("Год:", df["year"].notna().sum())
print("Размеры:", df["height_cm"].notna().sum())
print("Площадь:", df["area_cm2"].notna().sum())

Год: 115
Размеры: 14
Площадь: 14


### Задание 6. Вопросы к данным — 1,5 балла

Напишите 10 осмысленных запросов к пандас датафрейму. Каждый запрос должен отличаться механизмом поиска, то есть если вы найдёте все картины, написанные маслом, и все картины, написанные тушью, одним и тем же запросом, то это будет считаться за один запрос. Запросы должны сопровождаться пояснением (формата "Сколько уникальных художников в датафрейме?").

Каждый правильный ответ — **0,2 балла**. И удача любит смелых!

In [24]:
df["artist_short"].nunique()

103

In [25]:
df["date"].notna().sum()

np.int64(125)

In [30]:
df.groupby("classification").size().sort_values(ascending=False)

classification
Paintings    124
Drawings       1
dtype: int64

In [29]:
df[df["year"] > 1900][["title_clean", "artist_short", "year"]]

,title_clean,artist_short,year
46,Mäda Primavesi (1903–2000),Gustav Klimt,1912
109,The Repast of the Lion,Henri Rousseau,1907
117,Inferno,Franz von Stuck,1908


In [26]:
df["artist_short"].value_counts().head(10)

artist_short
Paul Cézanne           3
Edgar Degas            3
Vincent van Gogh       3
Edouard Manet          3
Claude Monet           3
Camille Corot          2
Gustave Courbet        2
Jacques Louis David    2
Anthony van Dyck       2
El Greco Toledo)       2
Name: count, dtype: int64

In [28]:
df[df["title_clean"].str.contains("Woman", case=False, na=False)]

,title,artist,date,medium,dimensions,classification,credit_line,object_number,department,url,title_clean,artist_short,year,height_cm,width_cm,area_cm2
22,Woman with a Parrot,"Gustave Courbet (French, Ornans 1819–1877 La T...",1866,Oil on canvas,51 x 77 in. (129.5 x 195.6 cm),Paintings,"H. O. Havemeyer Collection, Bequest of Mrs. H....",29.100.57,European Paintings,https://www.metmuseum.org/art/collection/searc...,Woman with a Parrot,Gustave Courbet,1866,<NA>,<NA>,<NA>
27,A Woman Seated beside a Vase of Flowers (Madam...,"Edgar Degas (French, Paris 1834–1917 Paris)",1865,Oil on canvas,29 x 36 1/2 in. (73.7 x 92.7 cm),Paintings,"H. O. Havemeyer Collection, Bequest of Mrs. H....",29.100.128,European Paintings,https://www.metmuseum.org/art/collection/searc...,A Woman Seated beside a Vase of Flowers (Madam...,Edgar Degas,1865,<NA>,<NA>,<NA>
28,Woman Having Her Hair Combed,"Edgar Degas (French, Paris 1834–1917 Paris)",ca. 1886–88,"Pastel on light green wove paper, now discolor...",29 1/8 x 23 7/8 in. (74 x 60.6 cm),Drawings,"H. O. Havemeyer Collection, Bequest of Mrs. H....",29.100.35,European Paintings,https://www.metmuseum.org/art/collection/searc...,Woman Having Her Hair Combed,Edgar Degas,1886,<NA>,<NA>,<NA>
52,Portrait of a Woman with a Man at a Casement,"Fra Filippo Lippi (Italian, Florence ca. 1406–...",ca. 1440,Tempera on wood,25 1/4 x 16 1/2 in. (64.1 x 41.9 cm),Paintings,"Marquand Collection, Gift of Henry G. Marquand...",89.15.19,European Paintings,https://www.metmuseum.org/art/collection/searc...,Portrait of a Woman with a Man at a Casement,Fra Filippo Lippi,1440,<NA>,<NA>,<NA>
65,Young Woman Seated on a Sofa,"Berthe Morisot (French, Bourges 1841–1895 Paris)",ca. 1879,Oil on canvas,31 3/4 × 39 1/4 in. (80.6 × 99.7 cm),Paintings,"Gift of Mr. and Mrs. Douglas Dillon, 1992",1992.103.2,European Paintings,https://www.metmuseum.org/art/collection/searc...,Young Woman Seated on a Sofa,Berthe Morisot,1879,<NA>,<NA>,<NA>
93,Study of a Young Woman,"Johannes Vermeer (Dutch, Delft 1632–1675 Delft)",ca. 1665–67,Oil on canvas,17 1/2 x 15 3/4 in. (44.5 x 40 cm),Paintings,"Gift of Mr. and Mrs. Charles Wrightsman, in me...",1979.396.1,European Paintings,https://www.metmuseum.org/art/collection/searc...,Study of a Young Woman,Johannes Vermeer,1665,<NA>,<NA>,<NA>
94,Young Woman with a Water Pitcher,"Johannes Vermeer (Dutch, Delft 1632–1675 Delft)",ca. 1662,Oil on canvas,18 x 16 in. (45.7 x 40.6 cm),Paintings,"Marquand Collection, Gift of Henry G. Marquand...",89.15.21,European Paintings,https://www.metmuseum.org/art/collection/searc...,Young Woman with a Water Pitcher,Johannes Vermeer,1662,<NA>,<NA>,<NA>


In [37]:
df["year"].agg(["min", "max"])

min    1290
max    1912
Name: year, dtype: int64

In [34]:
df[df["medium"].str.contains("oil", case=False, na=False)][
    ["title_clean", "artist_short", "medium"]
]

,title_clean,artist_short,medium
0,Saint John on Patmos,Hans Baldung 1484/85–1545 Strasbourg),"Oil, gold, and white metal on spruce"
1,Joan of Arc,Jules Bastien-Lepage,Oil on canvas
2,Madonna and Child,Giovanni Bellini,Oil on wood
3,The Horse Fair,Rosa Bonheur,Oil on canvas
5,The Toilette of Venus,François Boucher,Oil on canvas
...,...,...,...
119,A Bouquet of Flowers,Clara Peeters,Oil on wood
120,The Judgment of Paris,Lucas Cranach the Elder,Oil on beech
121,View of Toledo,El Greco Toledo),Oil on canvas
122,"Jalais Hill, Pontoise",Camille Pissarro,Oil on canvas


In [35]:
df["department"].value_counts()

department
European Paintings    125
Name: count, dtype: int64

In [46]:
df["height_cm"] = pd.to_numeric(df["height_cm"], errors="coerce")
df["width_cm"] = pd.to_numeric(df["width_cm"], errors="coerce")

df["area_cm2"] = df["height_cm"] * df["width_cm"]

In [47]:
print(df[["height_cm", "width_cm", "area_cm2"]].dtypes)

height_cm    float64
width_cm     float64
area_cm2     float64
dtype: object


In [48]:
df.nlargest(5, "area_cm2")[
    ["title_clean", "dimensions", "area_cm2"]
]

,title_clean,dimensions,area_cm2
72,Madonna and Child Enthroned with Saints,"Main panel, overall 67 7/8 x 67 7/8 in. (172.4...",29118.36
7,The Harvesters,"Overall, including added strips at top, bottom...",25839.00
12,The Meditation on the Passion,Overall 27 3/4 x 34 1/8in. (70.5 x 86.7cm); pa...,7326.15
0,Saint John on Patmos,Overall 35 1/4 x 30 1/4 in. (89.5 x 76.8 cm); ...,5806.08
35,The Triumph of Fame; (reverse) Impresa of the ...,"Overall, with engaged frame, diameter 36 1/2 i...",5793.75


## Часть 2. Selenium (2 балла)

Поработаем с каталогом музея Виктории и Альберта в Лондоне. Его страница поиска как раз
рисуется скриптом: в HTML лежит пустой контейнер, а карточки появляются
только в браузере.

In [61]:
%pip install selenium

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: C:\Users\honor\AppData\Local\Python\pythoncore-3.13-64\python.exe -m pip install --upgrade pip


In [69]:
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [70]:
SEARCH_URL = "https://collections.vam.ac.uk/search/?q=painting"
NEXT_SELECTOR = "button.b-search-pagination__next-link"

EXPECTED_PER_PAGE = 15
EXPECTED_TOTAL = 30

CARD_SELECTOR = ".b-search-results-row:not(.b-search-results__head)"

TITLE_SELECTOR = ".b-search-results-row__caption"
ARTIST_SELECTOR = ".b-search-results-row__subcaption"
DATE_SELECTOR = ".b-search-results-row__subcaption--2"

LINK_SELECTOR = "a[href*='/item/']:not(.b-search-results-row__img-pane)"

In [71]:
def create_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--window-size=1400,1000")

    return webdriver.Chrome(options=options)


driver = create_driver()
wait = WebDriverWait(driver, 40)

In [72]:
driver.get(SEARCH_URL)

wait.until(
    lambda d: len(
        d.find_elements(By.CSS_SELECTOR, CARD_SELECTOR)
    ) >= EXPECTED_PER_PAGE
)

cards = driver.find_elements(
    By.CSS_SELECTOR, CARD_SELECTOR
)

print("Карточек на 1 стр:", len(cards))

Карточек на 1 стр: 15


In [73]:
def text_or_empty(element, selector):
    try:
        return element.find_element(
            By.CSS_SELECTOR, selector
        ).text.strip()
    except:
        return ""


def parse_card(card):
    link_element = card.find_element(
        By.CSS_SELECTOR, LINK_SELECTOR
    )

    title = text_or_empty(
        link_element, TITLE_SELECTOR
    )

    artist = text_or_empty(
        link_element, ARTIST_SELECTOR
    )

    date = text_or_empty(
        link_element, DATE_SELECTOR
    )

    link = link_element.get_attribute("href")

    return {
        "title": title,
        "artist": artist,
        "date": date,
        "link": link
    }

In [76]:
def text_or_empty(element, selector):
    try:
        found_element = element.find_element(
            By.CSS_SELECTOR, selector
        )
        return found_element.get_attribute("textContent").strip()
    except:
        return ""

In [77]:
def parse_card(card):
    link_element = card.find_element(
        By.CSS_SELECTOR,
        "a[href*='/item/']:not(.b-search-results-row__img-pane)"
    )

    return {
        "title": text_or_empty(
            link_element, ".b-search-results-row__caption"
        ),
        "artist": text_or_empty(
            link_element, ".b-search-results-row__subcaption:not(.b-search-results-row__subcaption--2)"
        ),
        "date": text_or_empty(
            link_element, ".b-search-results-row__subcaption--2"
        ),
        "link": link_element.get_attribute("href")
    }

In [78]:
first = parse_card(cards[0])
print(first)

{'title': 'Krishna painting Radha', 'artist': 'Keyt, George', 'date': '1948', 'link': 'https://collections.vam.ac.uk/item/O429002/'}


In [79]:
records = [
    parse_card(card)
    for card in cards
]

print("Собран записей:", len(records))

Собран записей: 15


In [97]:
old_first_card = driver.find_element(
    By.CSS_SELECTOR, CARD_SELECTOR
)

next_button = wait.until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, NEXT_SELECTOR))
)

driver.execute_script(
    "arguments[0].scrollIntoView({block: 'center'});",
    next_button
)

next_button.click()

wait.until(EC.staleness_of(old_first_card))

wait.until(
    lambda d: "page=2" in d.current_url
)

cards_page_2 = driver.find_elements(
    By.CSS_SELECTOR, CARD_SELECTOR
)

print("URL:", driver.current_url)
print("Карточек на 2 стр:", len(cards_page_2))

URL: https://collections.vam.ac.uk/search/?page=2&page_size=15&q=painting
Карточек на 2 стр: 15


In [90]:
cookie_buttons = driver.find_elements(
    By.CSS_SELECTOR,
    "#onetrust-accept-btn-handler"
)

if cookie_buttons:
    driver.execute_script(
        "arguments[0].click();",
        cookie_buttons[0]
    )

    wait.until(
        EC.invisibility_of_element_located(
            (By.CSS_SELECTOR, ".onetrust-pc-dark-filter")
        )
    )


In [100]:
wait.until(
    lambda d: len(d.find_elements(
        By.CSS_SELECTOR, CARD_SELECTOR
    )) == 15
)

cards_page_2 = driver.find_elements(
    By.CSS_SELECTOR, CARD_SELECTOR
)

records_page_2 = [
    parse_card(card)
    for card in cards_page_2
]

print("Записей на 2 странице:", len(records_page_2))
print(records_page_2[:2])

Записей на 2 странице: 15
[{'title': 'One of nine paintings depicting Hindu deities.', 'artist': 'Unknown', 'date': 'ca. 1900', 'link': 'https://collections.vam.ac.uk/item/O432249/'}, {'title': 'Gujari Ragini', 'artist': 'Unknown', 'date': 'ca. 1760', 'link': 'https://collections.vam.ac.uk/item/O17467/'}]


In [101]:
all_records = records + records_page_2

df = pd.DataFrame(all_records)

print("Всего записей:", len(df))
display(df)

Всего записей: 90


,title,artist,date,link
0,Krishna painting Radha,"Keyt, George",1948,https://collections.vam.ac.uk/item/O429002/
1,Decorative painting for the Breakfast Room at ...,"Brunias, Agostino",1759-1761,https://collections.vam.ac.uk/item/O64940/
2,Decorative painting for Kedleston Hall,Agostino Brunias,1761,https://collections.vam.ac.uk/item/O53065/
3,Flower Painting,"Barwell, Frederick Bacon",1868,https://collections.vam.ac.uk/item/O155926/
4,Landscape Painting,"Leslie, George Dunlop",1868,https://collections.vam.ac.uk/item/O155923/
...,...,...,...,...
85,Copy of painting in the caves of Ajanta (cave 2),"Griffiths, John",1872-1885,https://collections.vam.ac.uk/item/O114063/
86,Copy of painting in the caves of Ajanta (Cave 2),"Griffiths, John",1872-1885,https://collections.vam.ac.uk/item/O117823/
87,Decorative painting for Kedleston Hall,"Brunias, Agostino",1759-1761,https://collections.vam.ac.uk/item/O65305/
88,Decorative painting for Kedleston Hall,"Brunias, Agostino",1759-1761,https://collections.vam.ac.uk/item/O65308/


In [102]:
print("Размер табл:", df.shape)

print("\nНазвания столбцов:")
print(df.columns.tolist())

print("\nКолво пропусков:")
print(df.isna().sum())

print("\nКол-во дубликатов:")
print(df.duplicated().sum())

Размер табл: (90, 4)

Названия столбцов:
['title', 'artist', 'date', 'link']

Колво пропусков:
title     0
artist    0
date      0
link      0
dtype: int64

Кол-во дубликатов:
45


In [103]:
display(df.head(10))

,title,artist,date,link
0,Krishna painting Radha,"Keyt, George",1948,https://collections.vam.ac.uk/item/O429002/
1,Decorative painting for the Breakfast Room at ...,"Brunias, Agostino",1759-1761,https://collections.vam.ac.uk/item/O64940/
2,Decorative painting for Kedleston Hall,Agostino Brunias,1761,https://collections.vam.ac.uk/item/O53065/
3,Flower Painting,"Barwell, Frederick Bacon",1868,https://collections.vam.ac.uk/item/O155926/
4,Landscape Painting,"Leslie, George Dunlop",1868,https://collections.vam.ac.uk/item/O155923/
5,Free House: Livery & Bait,Eric Gill,1929,https://collections.vam.ac.uk/item/O1240442/
6,Krishna,Unknown,ca. 1885,https://collections.vam.ac.uk/item/O109145/
7,Painting of a Court Official,unknown,1780-1880,https://collections.vam.ac.uk/item/O138049/
8,Oil painted study of an unidentified theatrica...,"Matania, Fortunino",early 20th century,https://collections.vam.ac.uk/item/O1305073/
9,Painting,Unknown,1600-1700,https://collections.vam.ac.uk/item/O251683/


In [106]:
print("Первая страница:", len(records))
print("Вторая страница:", len(records_page_2))
print("Всего:", len(records) + len(records_page_2))

Первая страница: 75
Вторая страница: 15
Всего: 90


In [107]:
df = pd.DataFrame(records + records_page_2)

df = df.drop_duplicates(subset="link").reset_index(drop=True)

print("Размер табл:", df.shape)
print("Кол-во дубликатов:", df.duplicated().sum())

display(df.head())

Размер табл: (45, 4)
Кол-во дубликатов: 0


,title,artist,date,link
0,Krishna painting Radha,"Keyt, George",1948,https://collections.vam.ac.uk/item/O429002/
1,Decorative painting for the Breakfast Room at ...,"Brunias, Agostino",1759-1761,https://collections.vam.ac.uk/item/O64940/
2,Decorative painting for Kedleston Hall,Agostino Brunias,1761,https://collections.vam.ac.uk/item/O53065/
3,Flower Painting,"Barwell, Frederick Bacon",1868,https://collections.vam.ac.uk/item/O155926/
4,Landscape Painting,"Leslie, George Dunlop",1868,https://collections.vam.ac.uk/item/O155923/


In [109]:
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 40)

driver.get(SEARCH_URL)

wait.until(
    lambda d: len(d.find_elements(By.CSS_SELECTOR, CARD_SELECTOR)) == 15
)

True

In [110]:
cards_page_1 = driver.find_elements(
    By.CSS_SELECTOR, CARD_SELECTOR
)

records_page_1 = [
    parse_card(card) for card in cards_page_1
]

print("Первая стр:", len(records_page_1))

Первая стр: 15


In [111]:
all_records = records_page_1 + records_page_2

df = pd.DataFrame(all_records)

print("Размер табл:", df.shape)
print("Дубликатов:", df.duplicated(subset="link").sum())

display(df.head())

Размер табл: (30, 4)
Дубликатов: 0


,title,artist,date,link
0,Krishna painting Radha,"Keyt, George",1948,https://collections.vam.ac.uk/item/O429002/
1,Decorative painting for the Breakfast Room at ...,"Brunias, Agostino",1759-1761,https://collections.vam.ac.uk/item/O64940/
2,Decorative painting for Kedleston Hall,Agostino Brunias,1761,https://collections.vam.ac.uk/item/O53065/
3,Flower Painting,"Barwell, Frederick Bacon",1868,https://collections.vam.ac.uk/item/O155926/
4,Landscape Painting,"Leslie, George Dunlop",1868,https://collections.vam.ac.uk/item/O155923/


In [112]:
df.to_csv("vam_paintings.csv", index=False, encoding="utf-8-sig")

print("Файл сохр")

Файл сохр


In [113]:
driver.quit()

In [ ]:
#Пока разбиралась с кодом случайно несколько раз добавляла карточки в список, исправила это, но свом попытки и ошибки тут оставила

## Справка: JavaScript —— зачем он нужен?

### HTML — это «скелет» страницы

Сервер присылает браузеру **HTML** — текстовую разметку: заголовки, абзацы,
списки, картинки, ссылки. HTML отвечает за **структуру и содержание**: что
и в каком порядке показать.

### JavaScript — это «поведение» страницы

**JavaScript** — язык программирования, который выполняется **в браузере**.
Скрипты сайта умеют:

- показать окно по нажатию кнопки;
- подставить данные, полученные отдельным запросом;
- отрисовать список, которого в HTML не было;
- отправить форму без перезагрузки страницы и тд.

Браузер выполняет скрипты **после** того, как получил HTML. Поэтому `requests`
(он просто скачивает HTML и ничего не выполняет) иногда видит «пустую» страницу,
а человек в браузере — полный список.

**DOM** — это «страница после работы скриптов»: дерево элементов, которое
браузер построил из HTML и изменил скриптами. Именно DOM, а не исходный HTML,
видит Selenium.

### Как понять, что страница «на JavaScript»

Сравните исходный код и DOM. В Chrome: `Ctrl+U` открывает исходный код — то, что прислал сервер.  **Elements** показывает DOM. Если в Elements есть блоки, которых нет в исходном коде, их добавил JavaScript.

## 5. Сколько баллов за что

| Задание | Что проверяем | Баллы |
|---|---|---|
| 1 | Функция `parse_artwork` разбирает страницу работы | 1 |
| 2 | Список из 125 абсолютных адресов, три страницы, без дублей | 2 |
| 3 | Краулер обошёл все работы с паузами, записи собраны | 2 |
| 4 | Посчитаны `artist_short`, `year`, размеры и площадь, есть DataFrame | 1 |
| 5 | Пять вопросов к данным посчитаны в pandas | 1 |
| 6 | CSV сохранён и прочитан, добавлена колонка, свой вопрос | 1 |
| | **Итого за основную часть** | **8** |
| 7 | Бонус: `requests` против Selenium на JS-странице V&A | +1 → 9 |
| 8 | Бонус: клик по Next и 30 записей в DataFrame | +1 → 10 |

**Как считается балл.** Задание засчитывается, если код запускается и в выводе
видны настоящие данные. Пустая таблица, «у меня не получилось» или код,
который падает с ошибкой, балла не даёт.

**Что точно не принимается.** Данные, вписанные руками; одна и та же картина,
скопированная в список; DataFrame, собранный не из результатов краулера, а из
файла, скачанного целиком. В заданиях 7–8 нужен именно Selenium: CSV/JSON API
музея вместо браузера не принимается.

## Шпаргалка: Selenium

| Что нужно | Как это сделать |
|---|---|
| Открыть страницу | `driver.get(url)` |
| Найти все элементы | `driver.find_elements(By.CSS_SELECTOR, 'div.card')` |
| Найти один элемент | `driver.find_element(By.CSS_SELECTOR, '#artwork-details')` |
| Дождаться элемента | `WebDriverWait(driver, 30).until(EC.presence_of_element_located((By.CSS_SELECTOR, '...')))` |
| Дождаться кликабельности | `EC.element_to_be_clickable((By.CSS_SELECTOR, '...'))` |
| Дождаться смены адреса | `EC.url_contains('page=2')` |
| Текст элемента | `element.text` или `element.get_attribute('textContent').strip()` |
| Нажать | `element.click()`; если перекрыт — `driver.execute_script('arguments[0].click();', element)` |
| Прокрутить к элементу | `driver.execute_script("arguments[0].scrollIntoView({block:'center'});", element)` |
| Текущий адрес | `driver.current_url` |
| Закрыть браузер | `driver.quit()` |